<a href="https://colab.research.google.com/github/ajrodloptfg06/TFG-DSM-DeepLearning/blob/main/TFG_DsmV6_FAST_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Version FAST_TEST

Este notebook esta pensado para pruebas cortas en Colab: usa `FAST_DEV_RUN = True`, entrena 2 epocas por modelo y desactiva pesos preentrenados para reducir descargas y consumo. Para resultados finales usa `TFG_DsmV6.ipynb`.


In [ ]:
!nvidia-smi

Mon Apr  6 10:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install --upgrade pip
!pip -q install timm einops albumentations torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.1 MB/s eta 0:00:00


In [ ]:
import os, random, time, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from einops import rearrange

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    #6seg

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/TFG_DSM"
CKPT_DIR = os.path.join(BASE, "checkpoints")
RES_DIR  = os.path.join(BASE, "results")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

print("CKPT_DIR:", CKPT_DIR)
print("RES_DIR:", RES_DIR)

Mounted at /content/drive
CKPT_DIR: /content/drive/MyDrive/TFG_DSM/checkpoints
RES_DIR: /content/drive/MyDrive/TFG_DSM/results


In [ ]:
FAST_DEV_RUN = True

CFG = {
    "in_channels": 4,
    "batch_size": 16,
    "num_workers": 2,
    "lr": 1e-4,
    "epochs": 2 if FAST_DEV_RUN else 50,
    "weight_decay": 1e-4,
    "pretrained_backbones": False if FAST_DEV_RUN else True,
    "ckpt_dir": CKPT_DIR,
    "results_dir": RES_DIR,
}
CFG


In [ ]:
import numpy as np

BASE = "/content/drive/MyDrive"

x_train = np.load(f"{BASE}/xtrain_nyc-002.npy")
y_train = np.load(f"{BASE}/ytrain_nyc.npy")
x_test  = np.load(f"{BASE}/xtest_nyc.npy")
y_test  = np.load(f"{BASE}/ytest_nyc.npy")

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

print("x_train min/max:", x_train.min(), x_train.max())
print("y_train min/max:", y_train.min(), y_train.max())


## 1 min

x_train: (6000, 128, 128, 4) float64
y_train: (6000, 128, 128, 1) float64
x_test: (330, 128, 128, 4)
y_test: (330, 128, 128, 1)
x_train min/max: -3.7899852752685548 16.885
y_train min/max: 0.0 405.8377990722656


In [ ]:
import numpy as np
import torch

# Convertimos a float32 (reduce memoria y acelera)
x_train_t = torch.from_numpy(x_train.astype(np.float32)).permute(0, 3, 1, 2)  # NHWC -> NCHW
y_train_t = torch.from_numpy(y_train.astype(np.float32)).permute(0, 3, 1, 2)  # (N,H,W,1) -> (N,1,H,W)

x_test_t  = torch.from_numpy(x_test.astype(np.float32)).permute(0, 3, 1, 2)
y_test_t  = torch.from_numpy(y_test.astype(np.float32)).permute(0, 3, 1, 2)

print("x_train_t:", x_train_t.shape, x_train_t.dtype)
print("y_train_t:", y_train_t.shape, y_train_t.dtype)
print("x_test_t:", x_test_t.shape)
print("y_test_t:", y_test_t.shape)

x_train_t: torch.Size([6000, 4, 128, 128]) torch.float32
y_train_t: torch.Size([6000, 1, 128, 128]) torch.float32
x_test_t: torch.Size([330, 4, 128, 128])
y_test_t: torch.Size([330, 1, 128, 128])


In [ ]:
def describe_tensor_dataset(x_train_t, y_train_t, x_test_t, y_test_t):
    def channel_stats(x, prefix):
        flat = x.permute(1, 0, 2, 3).reshape(x.shape[1], -1)
        rows = []
        for c in range(flat.shape[0]):
            values = flat[c]
            rows.append({
                "split": prefix,
                "channel": c,
                "mean": float(values.mean()),
                "std": float(values.std(unbiased=False)),
                "min": float(values.min()),
                "max": float(values.max()),
            })
        return rows

    y_rows = []
    for name, y in [("train", y_train_t), ("test", y_test_t)]:
        values = y.reshape(-1)
        y_rows.append({
            "split": name,
            "mean": float(values.mean()),
            "std": float(values.std(unbiased=False)),
            "min": float(values.min()),
            "max": float(values.max()),
        })

    x_stats = channel_stats(x_train_t, "train") + channel_stats(x_test_t, "test")
    return x_stats, y_rows

x_channel_stats, y_dsm_stats = describe_tensor_dataset(x_train_t, y_train_t, x_test_t, y_test_t)
print("Input channel stats:")
for row in x_channel_stats:
    print(row)
print("DSM target stats:")
for row in y_dsm_stats:
    print(row)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(x_train_t, y_train_t)

val_ratio = 0.1
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

test_loader  = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

train batches: 338
val batches: 38
test batches: 21


In [ ]:
import torch

@torch.no_grad()
def mae(pred, target):
    return torch.mean(torch.abs(pred - target))

@torch.no_grad()
def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))

@torch.no_grad()
def r2_score(pred, target):
    # R2 = 1 - SS_res/SS_tot
    y = target
    yhat = pred
    y_mean = torch.mean(y)
    ss_res = torch.sum((y - yhat) ** 2)
    ss_tot = torch.sum((y - y_mean) ** 2).clamp_min(1e-6)
    return 1.0 - ss_res / ss_tot

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

criterion = nn.SmoothL1Loss(beta=1.0)  # Huber

def _match_prediction_to_target(pred, target, model_name="model"):
    """Ensure prediction and target are comparable before loss/metrics."""
    if pred.ndim != target.ndim:
        raise ValueError(f"{model_name}: pred ndim {pred.ndim} != target ndim {target.ndim}")

    if pred.shape[0] != target.shape[0] or pred.shape[1] != target.shape[1]:
        raise ValueError(f"{model_name}: pred shape {tuple(pred.shape)} != target shape {tuple(target.shape)}")

    if pred.shape[-2:] != target.shape[-2:]:
        pred = F.interpolate(pred, size=target.shape[-2:], mode="bilinear", align_corners=False)

    if pred.shape != target.shape:
        raise ValueError(f"{model_name}: pred shape {tuple(pred.shape)} != target shape {tuple(target.shape)}")

    return pred

def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        pred = model(x)
        pred = _match_prediction_to_target(pred, y, model.__class__.__name__)

        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def evaluate_global(model, loader, return_loss=True):
    model.eval()

    abs_sum = 0.0
    sq_sum = 0.0
    loss_sum = 0.0
    batches = 0
    n = 0

    preds_all = []
    targets_all = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        pred = model(x)
        pred = _match_prediction_to_target(pred, y, model.__class__.__name__)

        diff = pred - y

        abs_sum += torch.sum(torch.abs(diff)).item()
        sq_sum += torch.sum(diff ** 2).item()
        n += y.numel()

        if return_loss:
            loss_sum += criterion(pred, y).item()
            batches += 1

        preds_all.append(pred.flatten().cpu())
        targets_all.append(y.flatten().cpu())

    mae = abs_sum / n
    rmse = (sq_sum / n) ** 0.5

    preds_all = torch.cat(preds_all)
    targets_all = torch.cat(targets_all)

    y_mean = torch.mean(targets_all)
    ss_res = torch.sum((targets_all - preds_all) ** 2)
    ss_tot = torch.sum((targets_all - y_mean) ** 2).clamp_min(1e-6)
    r2 = 1.0 - (ss_res / ss_tot)

    metrics = {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }
    if return_loss:
        metrics["loss"] = float(loss_sum / max(1, batches))
    return metrics


In [ ]:
import os, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def fit_model_resumable(model, train_loader, val_loader,
                        epochs=None, lr=None, weight_decay=None,
                        ckpt_path="last_checkpoint.pth",
                        best_ckpt_path=None):
    model = model.to(device)
    epochs = CFG["epochs"] if epochs is None else epochs
    lr = CFG["lr"] if lr is None else lr
    weight_decay = CFG["weight_decay"] if weight_decay is None else weight_decay

    if best_ckpt_path is None:
        root, ext = os.path.splitext(ckpt_path)
        best_ckpt_path = f"{root}_best{ext or '.pth'}"

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    start_epoch = 1
    best_rmse = float("inf")
    history = []

    # Reanudar siempre desde el ultimo checkpoint, no desde el mejor.
    if os.path.exists(ckpt_path):
        print("Reanudando desde checkpoint:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_rmse = ckpt.get("best_rmse", best_rmse)
        history = ckpt.get("history", [])
        print("   -> continua desde epoch", start_epoch)

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()

        tr_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = evaluate_global(model, val_loader)
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        dt = time.time() - t0

        row = {
            "epoch": epoch,
            "lr": lr_now,
            "train_loss": float(tr_loss),
            "val_loss": float(val_metrics["loss"]),
            "val_MAE": float(val_metrics["MAE"]),
            "val_RMSE": float(val_metrics["RMSE"]),
            "val_R2": float(val_metrics["R2"]),
        }
        history.append(row)

        print(f"Epoch {epoch:03d} | lr={lr_now:.2e} | train_loss={tr_loss:.4f} | "
              f"val_RMSE={val_metrics['RMSE']:.4f} | val_MAE={val_metrics['MAE']:.4f} | val_R2={val_metrics['R2']:.4f} | {dt:.1f}s")

        is_best = val_metrics["RMSE"] < best_rmse
        if is_best:
            best_rmse = val_metrics["RMSE"]

        last_state = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_rmse": best_rmse,
            "history": history,
            "best_ckpt_path": best_ckpt_path,
        }
        torch.save(last_state, ckpt_path)

        if is_best:
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "best_rmse": best_rmse,
                "history": history,
            }, best_ckpt_path)

    return history, best_rmse


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNAct(out_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class SwinUNet(nn.Module):
    def __init__(self, in_ch=4, encoder_name="swin_tiny_patch4_window7_224", pretrained=True, img_size=128):
        super().__init__()

        self.encoder = timm.create_model(
            encoder_name,
            pretrained=pretrained,
            features_only=True,
            in_chans=in_ch,
            img_size=img_size,  # <-- acepta 128x128
        )
        enc_chs = self.encoder.feature_info.channels()

        self.bottleneck = nn.Sequential(
            ConvBNAct(enc_chs[-1], 256),
            ConvBNAct(256, 256),
        )

        self.dec3 = DecoderBlock(256, enc_chs[-2], 256)
        self.dec2 = DecoderBlock(256, enc_chs[-3], 128)
        self.dec1 = DecoderBlock(128, enc_chs[-4], 64)

        self.head = nn.Sequential(
            ConvBNAct(64, 32),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, x):
      x_in_h, x_in_w = x.shape[-2], x.shape[-1]

      feats = self.encoder(x)  # lista de features NHWC

    # Convertir NHWC -> NCHW
      feats = [f.permute(0, 3, 1, 2) for f in feats]

      f1, f2, f3, f4 = feats

      x = self.bottleneck(f4)
      x = self.dec3(x, f3)
      x = self.dec2(x, f2)
      x = self.dec1(x, f1)

      x = self.head(x)
      x = F.interpolate(x, size=(x_in_h, x_in_w), mode="bilinear", align_corners=False)
      return x

In [ ]:
model = SwinUNet(in_ch=4, pretrained=CFG["pretrained_backbones"]).to(device)

x, y = next(iter(train_loader))
x = x.to(device)
with torch.no_grad():
    pred = model(x)

print("pred:", pred.shape, "target:", y.shape)


In [ ]:
last_swin_ckpt = os.path.join(CKPT_DIR, "last_swin_unet.pth")
best_swin_ckpt = os.path.join(CKPT_DIR, "best_swin_unet.pth")

model_swin = SwinUNet(in_ch=4, pretrained=CFG["pretrained_backbones"], img_size=128).to(device)

history_swin, best_rmse_swin = fit_model_resumable(
    model_swin,
    train_loader,
    val_loader,
    epochs=CFG["epochs"],
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    ckpt_path=last_swin_ckpt,
    best_ckpt_path=best_swin_ckpt,
)

best_rmse_swin

#45 min


In [ ]:
ckpt = torch.load(best_swin_ckpt, map_location=device)
model_swin.load_state_dict(ckpt["model"])

test_metrics_swin = evaluate_global(model_swin, test_loader)
test_metrics_swin


In [ ]:
results = []

results.append({
    "model": "Swin-UNet",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_loss": test_metrics_swin["loss"],
    "test_MAE": test_metrics_swin["MAE"],
    "test_RMSE": test_metrics_swin["RMSE"],
    "test_R2": test_metrics_swin["R2"],
})

results


In [ ]:
import pandas as pd
import os

df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_MAE,test_RMSE,test_R2
0,Swin-UNet,6.228235,4.957771,10.615047,0.30706


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class AttentionGate(nn.Module):
    """
    g = gating (del decoder)
    x = skip (del encoder)
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):
        # x: skip, g: decoder feature (resolución similar o se interpola)
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)

        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self, in_ch=4, base_ch=64):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_ch, base_ch)           # 128
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_ch, base_ch*2)       # 64
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(base_ch*2, base_ch*4)     # 32
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(base_ch*4, base_ch*8)     # 16
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base_ch*8, base_ch*16)  # 8

        # Decoder (upsample + attention + conv)
        self.up4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, kernel_size=2, stride=2)
        self.att4 = AttentionGate(F_g=base_ch*8, F_l=base_ch*8, F_int=base_ch*4)
        self.dec4 = DoubleConv(base_ch*16, base_ch*8)

        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, kernel_size=2, stride=2)
        self.att3 = AttentionGate(F_g=base_ch*4, F_l=base_ch*4, F_int=base_ch*2)
        self.dec3 = DoubleConv(base_ch*8, base_ch*4)

        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, kernel_size=2, stride=2)
        self.att2 = AttentionGate(F_g=base_ch*2, F_l=base_ch*2, F_int=base_ch)
        self.dec2 = DoubleConv(base_ch*4, base_ch*2)

        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, kernel_size=2, stride=2)
        self.att1 = AttentionGate(F_g=base_ch, F_l=base_ch, F_int=base_ch//2)
        self.dec1 = DoubleConv(base_ch*2, base_ch)

        # Head
        self.out = nn.Conv2d(base_ch, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        # Decoder + attention on skips
        d4 = self.up4(b)
        e4_att = self.att4(e4, d4)
        d4 = self.dec4(torch.cat([d4, e4_att], dim=1))

        d3 = self.up3(d4)
        e3_att = self.att3(e3, d3)
        d3 = self.dec3(torch.cat([d3, e3_att], dim=1))

        d2 = self.up2(d3)
        e2_att = self.att2(e2, d2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))

        d1 = self.up1(d2)
        e1_att = self.att1(e1, d1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))

        return self.out(d1)

In [ ]:
model_att = AttentionUNet(in_ch=4, base_ch=64).to(device)

x, y = next(iter(train_loader))
x = x.to(device)
with torch.no_grad():
    pred = model_att(x)

print("pred:", pred.shape, "target:", y.shape)

pred: torch.Size([16, 1, 128, 128]) target: torch.Size([16, 1, 128, 128])


In [ ]:
last_att_ckpt = os.path.join(CKPT_DIR, "last_attention_unet.pth")
best_att_ckpt = os.path.join(CKPT_DIR, "best_attention_unet.pth")

history_att, best_rmse_att = fit_model_resumable(
    model_att,
    train_loader,
    val_loader,
    epochs=CFG["epochs"],
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    ckpt_path=last_att_ckpt,
    best_ckpt_path=best_att_ckpt,
)

ckpt = torch.load(best_att_ckpt, map_location=device)
model_att.load_state_dict(ckpt["model"])

test_metrics_att = evaluate_global(model_att, test_loader)
test_metrics_att


In [ ]:
results.append({
    "model": "Attention-U-Net",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_loss": test_metrics_att["loss"],
    "test_MAE": test_metrics_att["MAE"],
    "test_RMSE": test_metrics_att["RMSE"],
    "test_R2": test_metrics_att["R2"],
})
results


In [ ]:


df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_loss,test_MAE,test_RMSE,test_R2
0,Swin-UNet,5.965727,4.527480,4.976559,8.626991,0.27469
1,Attention-U-Net,3.259811,3.647055,4.061402,7.357884,0.47027


In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class HRNetRegressor(nn.Module):
    def __init__(self, in_ch=4, backbone="hrnet_w18_small", pretrained=True):
        super().__init__()
        # features_only -> devuelve lista de features; el último suele tener buena resolución
        self.backbone = timm.create_model(
            backbone,
            pretrained=pretrained,
            features_only=True,
            in_chans=in_ch,
        )
        chs = self.backbone.feature_info.channels()
        last_ch = chs[-1]

        # Head de regresión (ligera)
        self.head = nn.Sequential(
            nn.Conv2d(last_ch, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, kernel_size=1)
        )

    def forward(self, x):
        H, W = x.shape[-2], x.shape[-1]
        feats = self.backbone(x)

        # timm a veces devuelve NHWC en algunos backbones; lo detectamos:
        f = feats[-1]
        if f.ndim == 4 and f.shape[1] != 128 and f.shape[-1] > 8:
            # heurística simple: si parece NHWC -> pasar a NCHW
            # (B,H,W,C) => (B,C,H,W)
            if f.shape[-1] > 8 and f.shape[1] < 8:
                f = f.permute(0, 3, 1, 2)

        y = self.head(f)

        # subir a tamaño original
        if y.shape[-2:] != (H, W):
            y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        return y

In [ ]:
model_hr = HRNetRegressor(in_ch=4, backbone="hrnet_w18_small", pretrained=CFG["pretrained_backbones"]).to(device)

x, y = next(iter(train_loader))
x = x.to(device)

with torch.no_grad():
    pred = model_hr(x)

print("pred:", pred.shape, "target:", y.shape)


In [ ]:
last_hr_ckpt = os.path.join(CKPT_DIR, "last_hrnet_w18_small.pth")
best_hr_ckpt = os.path.join(CKPT_DIR, "best_hrnet_w18_small.pth")

history_hr, best_rmse_hr = fit_model_resumable(
    model_hr,
    train_loader,
    val_loader,
    epochs=CFG["epochs"],
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    ckpt_path=last_hr_ckpt,
    best_ckpt_path=best_hr_ckpt,
)

ckpt = torch.load(best_hr_ckpt, map_location=device)
model_hr.load_state_dict(ckpt["model"])

test_metrics_hr = evaluate_global(model_hr, test_loader)
test_metrics_hr


In [ ]:
results.append({
    "model": "HRNet-W18-Small",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_loss": test_metrics_hr["loss"],
    "test_MAE": test_metrics_hr["MAE"],
    "test_RMSE": test_metrics_hr["RMSE"],
    "test_R2": test_metrics_hr["R2"],
})
results


In [ ]:

df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_loss,test_MAE,test_RMSE,test_R2
0,Swin-UNet,5.965727,4.527480,4.976559,8.626991,0.274690
1,Attention-U-Net,3.259811,3.647055,4.061402,7.357884,0.470270
2,HRNet-W18-Small,7.543494,5.237060,5.702252,9.829071,0.053117


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(conv => BN => ReLU) * 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    """Downscale with maxpool then double conv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    """Upscale then double conv"""
    def __init__(self, in_ch, skip_ch, out_ch, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
            self.conv = DoubleConv(in_ch + skip_ch, out_ch)
        else:
            self.up = nn.ConvTranspose2d(in_ch, in_ch, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=4, base_ch=64, bilinear=True, out_ch=1):
        super().__init__()
        self.inc = DoubleConv(in_ch, base_ch)              # 128
        self.down1 = Down(base_ch, base_ch*2)              # 64
        self.down2 = Down(base_ch*2, base_ch*4)            # 32
        self.down3 = Down(base_ch*4, base_ch*8)            # 16
        self.down4 = Down(base_ch*8, base_ch*16)           # 8

        self.up1 = Up(base_ch*16, base_ch*8,  base_ch*8,  bilinear=bilinear)
        self.up2 = Up(base_ch*8,  base_ch*4,  base_ch*4,  bilinear=bilinear)
        self.up3 = Up(base_ch*4,  base_ch*2,  base_ch*2,  bilinear=bilinear)
        self.up4 = Up(base_ch*2,  base_ch,    base_ch,    bilinear=bilinear)

        self.outc = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)

        return self.outc(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetPlusPlus(nn.Module):
    """
    U-Net++ (nested skip connections) para regresión DSM.
    Entrada: (B, in_ch, H, W)
    Salida:  (B, 1, H, W)
    """
    def __init__(self, in_ch=4, base_ch=64, out_ch=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Encoder convs (nivel i, j=0)
        self.conv00 = ConvBlock(in_ch, base_ch)
        self.conv10 = ConvBlock(base_ch, base_ch*2)
        self.conv20 = ConvBlock(base_ch*2, base_ch*4)
        self.conv30 = ConvBlock(base_ch*4, base_ch*8)
        self.conv40 = ConvBlock(base_ch*8, base_ch*16)

        # Decoder/nested convs (nivel i, j>0)
        self.conv01 = ConvBlock(base_ch + base_ch*2, base_ch)
        self.conv11 = ConvBlock(base_ch*2 + base_ch*4, base_ch*2)
        self.conv21 = ConvBlock(base_ch*4 + base_ch*8, base_ch*4)
        self.conv31 = ConvBlock(base_ch*8 + base_ch*16, base_ch*8)

        self.conv02 = ConvBlock(base_ch + base_ch + base_ch*2, base_ch)
        self.conv12 = ConvBlock(base_ch*2 + base_ch*2 + base_ch*4, base_ch*2)
        self.conv22 = ConvBlock(base_ch*4 + base_ch*4 + base_ch*8, base_ch*4)

        self.conv03 = ConvBlock(base_ch + base_ch + base_ch + base_ch*2, base_ch)
        self.conv13 = ConvBlock(base_ch*2 + base_ch*2 + base_ch*2 + base_ch*4, base_ch*2)

        self.conv04 = ConvBlock(base_ch + base_ch + base_ch + base_ch + base_ch*2, base_ch)

        self.outc = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def _up(self, x, ref):
        # upsample x a la resolución de ref
        return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        # Encoder
        x00 = self.conv00(x)
        x10 = self.conv10(self.pool(x00))
        x20 = self.conv20(self.pool(x10))
        x30 = self.conv30(self.pool(x20))
        x40 = self.conv40(self.pool(x30))

        # Nivel j=1
        x01 = self.conv01(torch.cat([x00, self._up(x10, x00)], dim=1))
        x11 = self.conv11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x21 = self.conv21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x31 = self.conv31(torch.cat([x30, self._up(x40, x30)], dim=1))

        # Nivel j=2
        x02 = self.conv02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))
        x12 = self.conv12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x22 = self.conv22(torch.cat([x20, x21, self._up(x31, x20)], dim=1))

        # Nivel j=3
        x03 = self.conv03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        x13 = self.conv13(torch.cat([x10, x11, x12, self._up(x22, x10)], dim=1))

        # Nivel j=4
        x04 = self.conv04(torch.cat([x00, x01, x02, x03, self._up(x13, x00)], dim=1))

        return self.outc(x04)

In [ ]:
@torch.no_grad()
def sanity_check_model_shapes(models, batch_size=2, in_channels=4, height=128, width=128):
    x = torch.randn(batch_size, in_channels, height, width, device=device)
    shape_report = []
    for name, model in models.items():
        model = model.to(device)
        model.eval()
        y = model(x)
        if y.shape[-2:] != (height, width):
            y = F.interpolate(y, size=(height, width), mode="bilinear", align_corners=False)
        ok = tuple(y.shape) == (batch_size, 1, height, width)
        shape_report.append({"model": name, "output_shape": tuple(y.shape), "ok": ok})
        if not ok:
            raise AssertionError(f"{name} devuelve {tuple(y.shape)}, esperado {(batch_size, 1, height, width)}")
    return shape_report

shape_report = sanity_check_model_shapes({
    "Swin-UNet": SwinUNet(in_ch=4, pretrained=False, img_size=128),
    "Attention-U-Net": AttentionUNet(in_ch=4, base_ch=64),
    "HRNet-W18-Small": HRNetRegressor(in_ch=4, backbone="hrnet_w18_small", pretrained=False),
    "U-Net": UNet(in_ch=4, base_ch=64),
    "U-Net++": UNetPlusPlus(in_ch=4, base_ch=64),
})
shape_report


In [ ]:
# U-Net
model_unet = UNet(in_ch=4, base_ch=64).to(device)
last_unet_ckpt = os.path.join(CKPT_DIR, "last_unet.pth")
best_unet_ckpt = os.path.join(CKPT_DIR, "best_unet.pth")
hist_unet, best_unet = fit_model_resumable(
    model_unet,
    train_loader,
    val_loader,
    epochs=CFG["epochs"],
    ckpt_path=last_unet_ckpt,
    best_ckpt_path=best_unet_ckpt,
)

# U-Net++
model_unetpp = UNetPlusPlus(in_ch=4, base_ch=64).to(device)
last_unetpp_ckpt = os.path.join(CKPT_DIR, "last_unetpp.pth")
best_unetpp_ckpt = os.path.join(CKPT_DIR, "best_unetpp.pth")
hist_unetpp, best_unetpp = fit_model_resumable(
    model_unetpp,
    train_loader,
    val_loader,
    epochs=CFG["epochs"],
    ckpt_path=last_unetpp_ckpt,
    best_ckpt_path=best_unetpp_ckpt,
)


In [ ]:
import os
import pandas as pd
import torch

# Rutas de checkpoints mejores, generados por fit_model_resumable(..., best_ckpt_path=...)
best_unet_ckpt = os.path.join(CKPT_DIR, "best_unet.pth")
best_unetpp_ckpt = os.path.join(CKPT_DIR, "best_unetpp.pth")

# ===== U-NET =====
ckpt = torch.load(best_unet_ckpt, map_location=device)
model_unet.load_state_dict(ckpt["model"])

test_metrics_unet = evaluate_global(model_unet, test_loader)

results.append({
    "model": "U-Net",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_loss": test_metrics_unet["loss"],
    "test_MAE": test_metrics_unet["MAE"],
    "test_RMSE": test_metrics_unet["RMSE"],
    "test_R2": test_metrics_unet["R2"],
})

# ===== U-NET++ =====
ckpt = torch.load(best_unetpp_ckpt, map_location=device)
model_unetpp.load_state_dict(ckpt["model"])

test_metrics_unetpp = evaluate_global(model_unetpp, test_loader)

results.append({
    "model": "U-Net++",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_loss": test_metrics_unetpp["loss"],
    "test_MAE": test_metrics_unetpp["MAE"],
    "test_RMSE": test_metrics_unetpp["RMSE"],
    "test_R2": test_metrics_unetpp["R2"],
})

# ===== GUARDAR RESULTADOS =====
df_results = pd.DataFrame(results)
results_path = os.path.join(RES_DIR, "results.csv")

df_results.to_csv(results_path, index=False)

print("Resultados actualizados y guardados en:", results_path)
df_results


In [ ]:
import os
import pandas as pd
import torch

def evaluate_best_checkpoints(model_specs, test_loader, results_path):
    rows = []
    for name, model, ckpt_path in model_specs:
        if not os.path.exists(ckpt_path):
            print(f"Saltando {name}: no existe {ckpt_path}")
            continue

        ckpt = torch.load(ckpt_path, map_location=device)
        model = model.to(device)
        model.load_state_dict(ckpt["model"])
        metrics = evaluate_global(model, test_loader)
        rows.append({
            "model": name,
            "checkpoint": os.path.basename(ckpt_path),
            "best_val_RMSE": float(ckpt["best_rmse"]),
            "test_loss": metrics["loss"],
            "test_MAE": metrics["MAE"],
            "test_RMSE": metrics["RMSE"],
            "test_R2": metrics["R2"],
        })

    df = pd.DataFrame(rows)
    df.to_csv(results_path, index=False)
    return df

results_path = os.path.join(RES_DIR, "results.csv")
df_results = evaluate_best_checkpoints([
    ("Swin-UNet", SwinUNet(in_ch=4, pretrained=False, img_size=128), os.path.join(CKPT_DIR, "best_swin_unet.pth")),
    ("Attention-U-Net", AttentionUNet(in_ch=4, base_ch=64), os.path.join(CKPT_DIR, "best_attention_unet.pth")),
    ("HRNet-W18-Small", HRNetRegressor(in_ch=4, backbone="hrnet_w18_small", pretrained=False), os.path.join(CKPT_DIR, "best_hrnet_w18_small.pth")),
    ("U-Net", UNet(in_ch=4, base_ch=64), os.path.join(CKPT_DIR, "best_unet.pth")),
    ("U-Net++", UNetPlusPlus(in_ch=4, base_ch=64), os.path.join(CKPT_DIR, "best_unetpp.pth")),
], test_loader, results_path)

print("Resultados regenerados desde checkpoints best en:", results_path)
df_results


In [ ]:
import random
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# ===== ELIGE EL MODELO QUE QUIERES VISUALIZAR =====
model_to_show = model_unet   # opciones: model_unet, model_unetpp, model_att, model_swin, model_hr
model_name = "U-Net"

# ===== MODO EVALUACION =====
model_to_show.eval()

# ===== ELEGIR UNA MUESTRA ALEATORIA DEL TEST =====
idx = random.randint(0, len(x_test_t) - 1)

x_sample = x_test_t[idx:idx+1].to(device)   # shape: (1, 4, 128, 128)
y_sample = y_test_t[idx:idx+1].to(device)   # shape: (1, 1, 128, 128)

# ===== PREDICCION =====
with torch.no_grad():
    pred = model_to_show(x_sample)
    pred = _match_prediction_to_target(pred, y_sample, model_name)

# ===== PASAR A NUMPY =====
x_np = x_sample[0].detach().cpu().numpy()     # (4, 128, 128)
pred_np = pred[0, 0].detach().cpu().numpy()   # (128, 128)
y_np = y_sample[0, 0].detach().cpu().numpy()  # (128, 128)
error = pred_np - y_np

sample_mae = float(abs(error).mean())
sample_rmse = float((error ** 2).mean() ** 0.5)

# Misma escala para prediccion y Ground Truth. Esto evita comparar mapas con colorbars distintas.
dsm_vmin = float(min(pred_np.min(), y_np.min()))
dsm_vmax = float(max(pred_np.max(), y_np.max()))
err_abs = float(max(abs(error.min()), abs(error.max())))

# ===== ERROR =====
plt.figure(figsize=(5, 4))
plt.imshow(error, cmap="coolwarm", vmin=-err_abs, vmax=err_abs)
plt.colorbar(label="Pred - GT")
plt.title(f"Error {model_name} | MAE={sample_mae:.2f} RMSE={sample_rmse:.2f}")
plt.axis("off")
plt.show()

# ===== VISUALIZACION =====
fig, axes = plt.subplots(1, 6, figsize=(22, 4))

for c in range(4):
    axes[c].imshow(x_np[c], cmap="gray")
    axes[c].set_title(f"Canal {c+1}")
    axes[c].axis("off")

im_pred = axes[4].imshow(pred_np, cmap="terrain", vmin=dsm_vmin, vmax=dsm_vmax)
axes[4].set_title(f"Prediccion\n{model_name}")
axes[4].axis("off")
plt.colorbar(im_pred, ax=axes[4], fraction=0.046, pad=0.04)

im_gt = axes[5].imshow(y_np, cmap="terrain", vmin=dsm_vmin, vmax=dsm_vmax)
axes[5].set_title("Ground Truth")
axes[5].axis("off")
plt.colorbar(im_gt, ax=axes[5], fraction=0.046, pad=0.04)

plt.suptitle(f"Muestra test {idx} | escala comun [{dsm_vmin:.2f}, {dsm_vmax:.2f}] | MAE={sample_mae:.2f} RMSE={sample_rmse:.2f}", fontsize=14)
plt.tight_layout()
plt.show()
